# Extract Modalink Frames (Colab) — **Second Google Account**

Use this when your first Drive is full. Run Colab **signed in with the new email** so frames are stored on that account’s Drive (not a second copy on the old one).

## Setup on the NEW Google account (do this in browser first)

1. Open Colab while logged into the **new email**
2. Open Modalink (shared link) → **Organize → Add shortcut to My Drive**  
   Link: https://drive.google.com/drive/folders/1ZXR5n4ry5RCdhPvOmb_ugf4sYc9FwdLH
3. Create a folder on the new My Drive, e.g. `My Drive/MasterData/egypt_modalink_frames`  
   (or share/add shortcut to your empty Data folder on the new account)
4. Have `annotations version 2.xlsx` ready to upload

## Space tip
- **Do not** extract again into the old full Drive
- This notebook defaults to saving frames under the **new** My Drive
- Optional: `STORE_MODE = "local_zip"` writes to Colab disk then you download a zip (uses almost no Drive space)

## Labels (recommended this time)
Default: **`Video Emotion (final)`** (better for face models).  
Set `LABEL_COL = "Final Overall (majority of modalities)"` only if you want the old labeling.

In [ ]:
# 1) Mount Drive of the account currently logged into Colab
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

from pathlib import Path
print("Logged-in My Drive top folders:")
for p in sorted(Path("/content/drive/MyDrive").iterdir())[:40]:
    print(" -", p.name)

In [ ]:
# 2) CONFIG — edit for the NEW account
from pathlib import Path

MYDRIVE = Path("/content/drive/MyDrive")

# After "Add shortcut to Drive", Modalink usually appears here:
MODALINK_ROOT = MYDRIVE / "Final Modalink Dataset"
# If shortcut is nested, set full path, e.g.:
# MODALINK_ROOT = MYDRIVE / "Master Documents and Drafts" / "Data" / "Final Modalink Dataset"

# Where NEW account stores frames (keep this on the new Drive only)
STORE_MODE = "drive"  # "drive" | "local_zip"

if STORE_MODE == "drive":
    FRAMES_ROOT = MYDRIVE / "MasterData" / "egypt_modalink_frames"
else:
    # Almost no Drive space used; download zip at the end
    FRAMES_ROOT = Path("/content/egypt_modalink_frames")

# Prefer face-relevant labels this run
LABEL_COL = "Video Emotion (final)"
# LABEL_COL = "Final Overall (majority of modalities)"  # old choice

ANNOTATIONS_XLSX = Path("/content/annotations_version_2.xlsx")

FRAMES_ROOT.mkdir(parents=True, exist_ok=True)
print("MODALINK_ROOT exists:", MODALINK_ROOT.exists(), "→", MODALINK_ROOT)
print("FRAMES_ROOT:", FRAMES_ROOT)
print("LABEL_COL:", LABEL_COL)
print("STORE_MODE:", STORE_MODE)

if not MODALINK_ROOT.exists():
    print("\nModalink not found. Searching My Drive for shortcuts...")
    for p in MYDRIVE.rglob("*"):
        if p.is_dir() and ("modalink" in p.name.lower() or p.name == "Final Modalink Dataset"):
            print(" Candidate:", p)

In [ ]:
# 3) Annotations Excel
from google.colab import files

if not ANNOTATIONS_XLSX.exists():
    hits = list(MYDRIVE.rglob("*annotations*version*2*.xlsx"))
    hits += list(MYDRIVE.rglob("*annotations version 2.xlsx"))
    if hits:
        ANNOTATIONS_XLSX = hits[0]
        print("Found on Drive:", ANNOTATIONS_XLSX)
    else:
        print("Upload annotations version 2.xlsx")
        up = files.upload()
        ANNOTATIONS_XLSX = Path("/content") / next(iter(up))
print("Using:", ANNOTATIONS_XLSX)

In [ ]:
# 4) Config + mapping
import re
import cv2
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

EMOTION_MAP = {
    "Anger": "angry",
    "Disgust": "disgust",
    "Fear": "fear",
    "Happiness": "happy",
    "Neutral": "neutral",
    "Sadness": "sad",
    "Surprise": "surprise",
}
SKIP_LABELS = {"", "nan", "none", "ambiguous", "amiguous"}

FPS_SAMPLE = 2.0
MAX_FACES_PER_SEGMENT = 15
MIN_FACE_SIZE = 60
FACE_PAD = 0.25
SAVE_SIZE = 96
DRY_RUN_LIMIT = None  # set 20 for a quick test

CASCADE = cv2.CascadeClassifier(str(Path(cv2.data.haarcascades) / "haarcascade_frontalface_default.xml"))
assert not CASCADE.empty()

df = pd.read_excel(ANNOTATIONS_XLSX)
print("Rows:", len(df))
print(df[LABEL_COL].value_counts(dropna=False))

In [ ]:
# 5) Build worklist

def normalize_label(raw):
    if pd.isna(raw):
        return None
    s = str(raw).strip()
    if s.lower() in SKIP_LABELS:
        return None
    return EMOTION_MAP.get(s)

def resolve_video_path(folder: str, video_file: str):
    rel = Path(str(folder)) / str(video_file)
    full = MODALINK_ROOT / rel
    if full.exists():
        return full
    folder_dir = MODALINK_ROOT / str(folder)
    name = Path(str(video_file)).name
    if folder_dir.exists():
        hits = list(folder_dir.rglob(name))
        if hits:
            return hits[0]
    return None

rows, skipped = [], {"no_label": 0, "missing_video": 0}
for i, r in df.iterrows():
    emotion = normalize_label(r.get(LABEL_COL))
    if emotion is None:
        skipped["no_label"] += 1
        continue
    folder = str(r["Folder"]).strip()
    video_file = str(r["video_file"]).strip()
    speaker = str(r["speaker"]).strip()
    vpath = resolve_video_path(folder, video_file)
    if vpath is None:
        skipped["missing_video"] += 1
        continue
    rows.append({
        "excel_row": int(i),
        "folder": folder,
        "speaker": speaker,
        "person_id": f"{folder}|{speaker}",
        "emotion": emotion,
        "video_path": str(vpath),
        "segment_id": r.get("segment_id"),
        "speaker_segment_id": r.get("speaker_segment_id"),
    })

work = pd.DataFrame(rows)
if DRY_RUN_LIMIT:
    work = work.head(DRY_RUN_LIMIT)
print("Usable segments:", len(work), "| Skipped:", skipped)
print(work["emotion"].value_counts())

In [ ]:
# 6) Face extract helpers

def largest_face_bgr(frame_bgr):
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    faces = CASCADE.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(MIN_FACE_SIZE, MIN_FACE_SIZE))
    if len(faces) == 0:
        return None
    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
    pad_x, pad_y = int(w * FACE_PAD), int(h * FACE_PAD)
    x1, y1 = max(0, x - pad_x), max(0, y - pad_y)
    x2, y2 = min(frame_bgr.shape[1], x + w + pad_x), min(frame_bgr.shape[0], y + h + pad_y)
    return frame_bgr[y1:y2, x1:x2]

def safe_stem(person_id, emotion, segment_id):
    pid = re.sub(r"[^A-Za-z0-9_|-]+", "_", person_id)
    return f"{pid}__{emotion}__seg{segment_id}"

def extract_faces_from_video(video_path: Path, out_dir: Path, stem: str):
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    step = max(1, int(round(fps / FPS_SAMPLE)))
    saved, idx, kept = [], 0, 0
    while kept < MAX_FACES_PER_SEGMENT:
        ok, frame = cap.read()
        if not ok:
            break
        if idx % step == 0:
            face = largest_face_bgr(frame)
            if face is not None and face.size > 0:
                face = cv2.resize(face, (SAVE_SIZE, SAVE_SIZE), interpolation=cv2.INTER_AREA)
                out_path = out_dir / f"{stem}__f{kept:03d}.jpg"
                cv2.imwrite(str(out_path), face, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
                saved.append(str(out_path))
                kept += 1
        idx += 1
    cap.release()
    return saved

In [ ]:
# 7) Run extraction
import json

manifest_rows, n_images, n_empty = [], 0, 0
for _, row in tqdm(work.iterrows(), total=len(work)):
    emotion, person_id = row["emotion"], row["person_id"]
    out_dir = FRAMES_ROOT / "by_emotion" / emotion / person_id.replace("|", "__")
    stem = safe_stem(person_id, emotion, row["speaker_segment_id"])
    paths = extract_faces_from_video(Path(row["video_path"]), out_dir, stem)
    if not paths:
        n_empty += 1
        continue
    for p in paths:
        manifest_rows.append({
            "image_path": p,
            "emotion": emotion,
            "person_id": person_id,
            "folder": row["folder"],
            "speaker": row["speaker"],
            "video_path": row["video_path"],
            "excel_row": row["excel_row"],
            "label_col": LABEL_COL,
        })
        n_images += 1

manifest = pd.DataFrame(manifest_rows)
manifest_path = FRAMES_ROOT / "manifest.csv"
manifest.to_csv(manifest_path, index=False)

summary = {
    "label_col": LABEL_COL,
    "store_mode": STORE_MODE,
    "n_segments": int(len(work)),
    "n_images": int(n_images),
    "n_no_face": int(n_empty),
    "emotion_counts": manifest["emotion"].value_counts().to_dict() if len(manifest) else {},
    "n_persons": int(manifest["person_id"].nunique()) if len(manifest) else 0,
    "frames_root": str(FRAMES_ROOT),
}
with open(FRAMES_ROOT / "extract_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print("Manifest:", manifest_path)

In [ ]:
# 8) If STORE_MODE == local_zip → zip + download (saves Drive space)
from google.colab import files
import shutil

if STORE_MODE == "local_zip":
    zip_path = Path("/content/egypt_modalink_frames_zip")
    shutil.make_archive(str(zip_path), "zip", FRAMES_ROOT)
    print("Downloading zip...")
    files.download(str(zip_path) + ".zip")
    print("Upload this zip later to Kaggle as a dataset, or to the new Drive manually.")
else:
    print("Frames are on the NEW account Drive at:")
    print(FRAMES_ROOT)
    print("Share that folder with your main email if needed for later notebooks.")